In [ ]:
#
# Universidad EAFIT
# 2026-2
# SI7016 - NLP - Lecture 05b (actualizado 2026-2)
#

# 1. Construir un sistema RAG que permita buscar información en una base de datos real de noticias, recuperando los artículos más relevantes y generando respuestas con un LLM de frontera.

* Usa LangChain y ChromaDB para la recuperación (embeddings de OpenAI).
* Procesa un dataset real de noticias.
* Genera respuestas basadas en noticias relevantes.
* Evalúa la precisión con métricas como Recall@K y ROUGE (ver también la nota sobre RAGAS al final).

In [1]:
# 1. Instalación de dependencias
%pip install langchain-openai langchain-chroma chromadb pandas tiktoken rouge_score

  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 125.1/125.1 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 73.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 22.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 91.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.1/23.1 MB 75.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.2/137.2 kB 13.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.6/204.6 kB 18.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.7/95.7 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 5.9 MB/s eta 0:00:00
   ━━━━━━

# 2 - Descarga y Preprocesamiento del Dataset

Usaremos el dataset "News Category Dataset" de Kaggle, que contiene noticias reales.

Descarga el dataset desde Kaggle:

https://www.kaggle.com/datasets/rmisra/news-category-dataset?resource=download

descarga el archivo: News_Category_Dataset_v3.json.zip (déjalo en la misma carpeta que este notebook - la siguiente celda lo descomprime automáticamente si hace falta).

# Cargar y Preprocesar el Dataset

El dataset de noticias viene en formato JSON, por lo que lo cargaremos y seleccionaremos las columnas necesarias.

In [3]:
import pandas as pd
import json
import zipfile
import os

# Descomprimir el dataset si aún no existe el .json (el .zip se descarga de Kaggle)
file_path = "News_Category_Dataset_v3.json"
zip_path = "News_Category_Dataset_v3.json.zip"
if not os.path.exists(file_path) and os.path.exists(zip_path):
    with zipfile.ZipFile(zip_path) as z:
        z.extractall(".")

# Leer el archivo JSON línea por línea
data = []
with open(file_path, "r") as file:
    for line in file:
        data.append(json.loads(line))

# Convertir a un DataFrame de pandas
df = pd.DataFrame(data)

# Seleccionar columnas relevantes
df = df[['headline', 'short_description', 'category']]
df = df.dropna()  # Eliminar filas con valores nulos

# Crear una columna combinada con el título y la descripción para indexación
df["content"] = df["headline"] + ". " + df["short_description"]

print(df.head())

# volverlo más pequeño para poderlo procesar en tiempos razonables de clase y en una laptop
df = df[:10000]

                                            headline  \
0  Over 4 Million Americans Roll Up Sleeves For O...   
1  American Airlines Flyer Charged, Banned For Li...   
2  23 Of The Funniest Tweets About Cats And Dogs ...   
3  The Funniest Tweets From Parents This Week (Se...   
4  Woman Who Called Cops On Black Bird-Watcher Lo...   

                                   short_description   category  \
0  Health experts said it is too early to predict...  U.S. NEWS   
1  He was subdued by passengers and crew when he ...  U.S. NEWS   
2  "Until you have a dog you don't understand wha...     COMEDY   
3  "Accidentally put grown-up toothpaste on my to...  PARENTING   
4  Amy Cooper accused investment firm Franklin Te...  U.S. NEWS   

                                             content  
0  Over 4 Million Americans Roll Up Sleeves For O...  
1  American Airlines Flyer Charged, Banned For Li...  
2  23 Of The Funniest Tweets About Cats And Dogs ...  
3  The Funniest Tweets From Parents This

In [4]:
print(df.describe())

                                                 headline  \
count                                               10000   
unique                                               9967   
top     What To Watch On Amazon Prime That’s New This ...   
freq                                                   13   

                                        short_description  category  \
count                                               10000     10000   
unique                                               9995        29   
top     The 25 most profound “Shower Thoughts” on Redd...  POLITICS   
freq                                                    3      3361   

                                                  content  
count                                               10000  
unique                                               9999  
top     Netanyahu Holds Solid Lead In Israeli Election...  
freq                                                    2  


# 3. Crear la Base de Datos Vectorial en ChromaDB

Usamos ChromaDB (a través de `langchain_chroma`) para almacenar y recuperar embeddings de noticias.

In [5]:
import os
from getpass import getpass
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma

if "OPENAI_API_KEY" not in os.environ:
    os.environ["OPENAI_API_KEY"] = getpass("OpenAI API Key: ")

# Un único cliente (langchain_chroma.Chroma) para escribir y leer.
# Nota 2026: la versión original de este notebook creaba un
# chromadb.PersistentClient "crudo" y llamaba collection.add(...) sin pasarle
# el embedding_function de OpenAI que se había instanciado - chromadb usaba
# en silencio su propio embedding por defecto (sentence-transformers), no el
# de OpenAI. Aquí embeddings y almacenamiento pasan por el mismo objeto.
embedding_model = OpenAIEmbeddings(model="text-embedding-3-large")
vectorstore = Chroma(
    collection_name="news_articles",
    embedding_function=embedding_model,
    persist_directory="./chroma_news_db",
)

texts = df["content"].tolist()
ids = [str(idx) for idx in df.index]
metadatas = [
    {"category": row["category"], "headline": row["headline"]}
    for _, row in df.iterrows()
]

# Insertar en lotes: reduce drásticamente el número de llamadas a la API de
# embeddings frente a un insert por fila (el notebook original tardaba
# ~1 hora para 10.000 registros llamando a la API una fila a la vez).
batch_size = 100
for start in range(0, len(texts), batch_size):
    end = start + batch_size
    vectorstore.add_texts(
        texts[start:end],
        metadatas=metadatas[start:end],
        ids=ids[start:end],
    )
    if (start // batch_size) % 10 == 0:
        print(start)

print("Base de datos de noticias creada con éxito.")

OpenAI API Key: ··········
0
1000
2000
3000
4000
5000
6000
7000
8000
9000
Base de datos de noticias creada con éxito.


Inicializamos ChromaDB en modo persistente (vía `langchain_chroma`).
Generamos embeddings con OpenAI (`text-embedding-3-large`) y almacenamos los textos por lotes.
Guardamos metadatos como categoría y título para mejorar la interpretación de los resultados.

# 4. Búsqueda Semántica con Recuperación de Noticias

Ahora creamos una función para buscar las noticias más relevantes usando búsqueda vectorial en ChromaDB.

In [6]:
def search_news(query, top_k=3):
    results = vectorstore.similarity_search(query, k=top_k)
    docs = [doc.page_content for doc in results]
    metadata = [doc.metadata for doc in results]
    return docs, metadata

# Prueba de búsqueda
query = "News about Trump, white house"
retrieved_docs, metadata = search_news(query)

print("\nNoticias recuperadas:")
for i in range(len(retrieved_docs)):
    print(f"\n{metadata[i]['headline']}")
    print(f"\n{retrieved_docs[i]}")


Noticias recuperadas:

No White House Progress On Day 3 Of Government Shutdown

No White House Progress On Day 3 Of Government Shutdown. “Nothing new. Nothing new on the shutdown. Nothing new. Except we need border security,” Trump told reporters at the White House

Trump's Silent Public Outing Belies White House In Tumult

Trump's Silent Public Outing Belies White House In Tumult. The president's appearance at Arlington National Cemetery was his first public outing for official business in more than a week.

Trump Is Basking In Surreal, Adoring Mar-a-Lago Bubble, Says Reporter

Trump Is Basking In Surreal, Adoring Mar-a-Lago Bubble, Says Reporter. Everybody there "loves him," a Washington Post journalist revealed after interviewing Trump at his resort home.


`search_news()` consulta ChromaDB para recuperar los artículos más relevantes.
Ejemplo de consulta: buscamos noticias sobre Trump/la Casa Blanca y mostramos los títulos y descripciones.

# 5. Generación de Respuestas con un LLM de frontera

Integramos ahora la recuperación con un LLM (2026: `gpt-5.6`) para responder preguntas basadas en las noticias.

In [7]:
from langchain_openai import ChatOpenAI

# Configurar el modelo de lenguaje
llm = ChatOpenAI(model="gpt-5.6")

# Crear la función de RAG
def rag_query(user_query, top_k=3):
    retrieved_docs, metadata = search_news(user_query, top_k)

    # Formatear el contexto para el modelo
    context = "\n".join(retrieved_docs)

    # Crear el prompt para el modelo
    prompt = f"""
    Basado en las siguientes noticias, responde la siguiente pregunta de manera clara y concisa:

    Noticias:
    {context}

    Pregunta: {user_query}
    """

    # Generar la respuesta
    response = llm.invoke(prompt)

    return response.content

# Prueba de generación
user_query = "¿Cuáles son las últimas noticias sobre el cambio climático?"
response = rag_query(user_query)

print("\nRespuesta generada por RAG:")
print(response)


Respuesta generada por RAG:
Las últimas noticias advierten que el mundo no está en camino de cumplir el Acuerdo de París y que el objetivo de limitar el calentamiento global a 1,5 °C está “en soporte vital”. Un nuevo informe de la ONU señala que aún existe una vía estrecha para evitar una catástrofe climática, pero exige medidas inmediatas. António Guterres también alertó de que la voluntad política se está debilitando, mientras la guerra de Rusia dificulta aún más la acción climática.


Usamos un LLM de frontera para generar respuestas basadas en las noticias recuperadas.
`rag_query()` recupera noticias, formatea un prompt y consulta al modelo.
Prueba con consulta real: "¿Cuáles son las últimas noticias sobre el cambio climático?"

# 6. Evaluación del Sistema RAG

Evaluamos la recuperación y generación con Recall@K y ROUGE (y vemos por qué esto no basta en 2026 - nota al final).

## 6.1 Evaluación de la Recuperación con Recall@K

In [8]:
def evaluate_recall(queries, ground_truths, k=3):
    retrieved_texts = [search_news(q, k)[0] for q in queries]
    recall_k = sum(any(gt in retrieved for retrieved in retrieved_texts) for gt in ground_truths) / len(queries)

    print(f"\nRecall {k}: {recall_k:.2f}")

# Definir consultas y respuestas esperadas
test_queries = ["Noticias sobre economía", "Últimos eventos deportivos"]
expected_answers = ["Economía en crecimiento", "Partido de fútbol"]

evaluate_recall(test_queries, expected_answers, k=3)


Recall 3: 0.00


Recall@K mide la capacidad del sistema de recuperar información correcta.
Si el recall es alto (cercano a 1), la recuperación es precisa.

**Nota:** esta prueba compara por coincidencia exacta de subcadena, así que sirve para ilustrar el concepto pero no como benchmark real (ver RAGAS en la nota final para un enfoque más riguroso).

## 6.2 Evaluación de la Generación con ROUGE

In [9]:
from rouge_score import rouge_scorer

# Comparar respuestas generadas vs. esperadas
reference = "El cambio climático está afectando las temperaturas globales."
generated = rag_query("¿Cómo afecta el cambio climático al planeta?")

scorer = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=True)
scores = scorer.score(reference, generated)

print("\nMétricas de Evaluación de Generación:")
print(scores)


Métricas de Evaluación de Generación:
{'rouge1': Score(precision=0.10869565217391304, recall=0.5555555555555556, fmeasure=0.1818181818181818), 'rouge2': Score(precision=0.06666666666666667, recall=0.375, fmeasure=0.11320754716981134), 'rougeL': Score(precision=0.10869565217391304, recall=0.5555555555555556, fmeasure=0.1818181818181818)}


ROUGE mide la superposición de palabras entre la respuesta generada y una referencia escrita a mano.
Un ROUGE alto sugiere que la respuesta se parece a la referencia - pero no dice si la respuesta está *fundamentada* en las noticias recuperadas (ver nota de RAGAS más abajo).

# Conclusiones

* Creamos un sistema RAG real con ChromaDB (embeddings de OpenAI) para búsqueda de noticias.
* Probamos consultas reales y generamos respuestas con un LLM de frontera.
* Evaluamos la recuperación con Recall@K y la generación con ROUGE.

## Nota 2026: RAGAS en vez de (o junto a) BLEU/ROUGE

BLEU y ROUGE comparan contra una referencia humana, pero no miden si la respuesta está fundamentada en los documentos recuperados. El estándar 2026 para evaluar sistemas RAG es el framework **RAGAS**, con métricas centradas en RAG:

- **Faithfulness:** ¿las afirmaciones de la respuesta están respaldadas por el contexto recuperado?
- **Answer Relevancy:** ¿la respuesta contesta la pregunta del usuario?
- **Context Precision / Context Recall:** ¿los fragmentos relevantes fueron recuperados, y aparecen primero?

RETO: (1) actualización automática de noticias - (2) visualización de embeddings - (3) evaluar este mismo pipeline con RAGAS y comparar contra los resultados de Recall@K/ROUGE de arriba.